# Transformer + SAC pour station EV/PV/Grid/Batterie (Colab)

Notebook **100% autonome** : copie/colle directement dans Google Colab et exécute cellule par cellule.

Objectif:
- Entraîner un agent **SAC avec extracteur Transformer** pendant 10 épisodes.
- Afficher les métriques d'entraînement.
- Tracer les graphiques **reward** et **cost** par épisode.


In [ ]:
# (Optionnel) Installation si nécessaire dans Colab
# Décommente si torch n'est pas déjà disponible
# !pip install -q torch matplotlib numpy


In [ ]:
import math
import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__)
print('Device:', device)


In [ ]:
class ChargingStationEnv:
    def __init__(self, pv_profile, grid_prices, scenario=None):
        self.pv_profile = pv_profile
        self.grid_prices = grid_prices
        self.scenario = scenario

        self.BATT_CAPACITY = 150.0
        self.BATT_MAX_POWER = 35.0
        self.PV_MAX_POWER = 60.0
        self.EV_MAX_CH_POWER = 11.0
        self.GRID_EXPORT_PRICE = 0.05

        self.reset()

    def reset(self, seed=None):
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)

        self.current_step = 0
        self.batt_soc = 45.0
        self.cumulative_cost = 0.0
        self.ep_pv_used = 0.0
        self.ep_pv_avail = 0.0
        self.ep_grid_used = 0.0
        self.ep_total_demand = 0.0

        if self.scenario is not None:
            import copy
            self.active_evs = copy.deepcopy(self.scenario)
        else:
            self.active_evs = []
            total_hours = len(self.pv_profile)
            for h in range(total_hours):
                num_arrivals = np.random.poisson(1.0)
                for _ in range(num_arrivals):
                    cap = np.random.uniform(50, 100)
                    init_soc = np.clip(np.random.normal(0.20, 0.15), 0.05, 0.50)
                    target_soc = np.clip(np.random.normal(0.80, 0.20), 0.60, 1.00)
                    duration = np.random.randint(4, 13)
                    self.active_evs.append({
                        'arrival_time': h,
                        'initial_soc': float(init_soc),
                        'target_soc': float(target_soc),
                        'capacity': float(cap),
                        'duration': int(duration),
                        'departure_time': int(h + duration)
                    })

        for ev in self.active_evs:
            ev['current_soc'] = ev['initial_soc']

        return self._get_obs(), {}

    def _get_obs(self):
        hour = self.current_step % 24
        pv = self.pv_profile[self.current_step]
        grid_price = self.grid_prices[self.current_step]

        ev_needed = sum(
            max(0.0, ev['target_soc'] - ev['current_soc']) * ev['capacity']
            for ev in self.active_evs
            if ev['arrival_time'] <= self.current_step < ev['departure_time']
        )
        return np.array([pv, self.batt_soc, grid_price, hour, ev_needed], dtype=np.float32)

    def step(self, action):
        batt_action = float(np.clip(action[0], -1.0, 1.0))
        ev_action = float(np.clip(action[1], 0.0, 1.0))

        pv_avail = self.pv_profile[self.current_step]
        grid_price = self.grid_prices[self.current_step]

        present_evs = [
            ev for ev in self.active_evs
            if ev['arrival_time'] <= self.current_step < ev['departure_time']
        ]
        total_ev_demand = len(present_evs) * self.EV_MAX_CH_POWER * ev_action

        reward = 0.0

        pv_to_ev = min(pv_avail, total_ev_demand)
        pv_remaining = pv_avail - pv_to_ev
        ev_remaining_demand = total_ev_demand - pv_to_ev
        reward += pv_to_ev * 2.0

        batt_room = self.BATT_CAPACITY - self.batt_soc
        pv_to_batt = min(pv_remaining, batt_room, self.BATT_MAX_POWER)
        pv_remaining -= pv_to_batt
        self.batt_soc += pv_to_batt
        reward += pv_to_batt * 1.0

        grid_export = pv_remaining
        reward += grid_export * self.GRID_EXPORT_PRICE

        batt_avail = max(0.0, self.batt_soc)
        batt_to_ev = min(batt_avail, ev_remaining_demand, self.BATT_MAX_POWER)
        self.batt_soc -= batt_to_ev
        ev_remaining_demand -= batt_to_ev
        reward += batt_to_ev * 0.3

        grid_to_ev = ev_remaining_demand
        reward -= grid_to_ev * grid_price

        grid_to_batt = 0.0
        if batt_action > 0:
            charge_power = min(batt_action * self.BATT_MAX_POWER, self.BATT_CAPACITY - self.batt_soc)
            self.batt_soc += charge_power
            reward -= charge_power * grid_price
            grid_to_batt = charge_power
            if grid_price <= 0.11:
                reward += charge_power * 0.1
        elif batt_action < 0:
            discharge_power = min(abs(batt_action) * self.BATT_MAX_POWER, self.batt_soc)
            self.batt_soc -= discharge_power
            reward += discharge_power * self.GRID_EXPORT_PRICE

        for ev in self.active_evs:
            if ev['departure_time'] == self.current_step and ev['current_soc'] < ev['target_soc']:
                shortfall_kwh = (ev['target_soc'] - ev['current_soc']) * ev['capacity']
                reward -= shortfall_kwh * 10.0

        if present_evs:
            total_energy_delivered = pv_to_ev + batt_to_ev + grid_to_ev
            not_full_evs = [ev for ev in present_evs if ev['current_soc'] < ev['target_soc']]
            if not_full_evs:
                energy_per_ev = total_energy_delivered / len(not_full_evs)
                for ev in not_full_evs:
                    ev['current_soc'] = min(1.0, ev['current_soc'] + energy_per_ev / ev['capacity'])

        self.current_step += 1
        done = self.current_step >= len(self.pv_profile) - 1

        hourly_cost = (grid_to_ev + grid_to_batt) * grid_price - (grid_export * self.GRID_EXPORT_PRICE)
        self.cumulative_cost += hourly_cost

        self.ep_pv_used += pv_to_ev + pv_to_batt
        self.ep_pv_avail += pv_avail
        self.ep_grid_used += grid_to_ev + grid_to_batt
        self.ep_total_demand += pv_to_ev + batt_to_ev + grid_to_ev + grid_to_batt

        info = {
            'pv_to_ev': pv_to_ev,
            'batt_to_ev': batt_to_ev,
            'grid_to_ev': grid_to_ev,
            'grid_to_batt': grid_to_batt,
            'batt_soc': self.batt_soc,
            'grid_export': grid_export,
            'pv_to_batt': pv_to_batt,
            'total_ev_demand': total_ev_demand,
            'grid_price': grid_price,
            'hourly_cost': hourly_cost,
            'cumulative_cost': self.cumulative_cost,
        }

        if done:
            fulfilled_evs = sum(1 for ev in self.active_evs if ev['current_soc'] >= ev['target_soc'] - 0.01)
            info['satisfaction'] = (fulfilled_evs / len(self.active_evs) * 100.0) if self.active_evs else 100.0
            info['pv_usage_pct'] = (self.ep_pv_used / self.ep_pv_avail * 100.0) if self.ep_pv_avail > 0 else 0.0
            info['grid_dependency_pct'] = (self.ep_grid_used / self.ep_total_demand * 100.0) if self.ep_total_demand > 0 else 0.0

        return self._get_obs(), reward, done, False, info


In [ ]:
class TransformerFeatureExtractor(nn.Module):
    def __init__(self, input_dim, d_model=128, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True,
            dropout=dropout,
            activation='gelu',
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)

    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        return x[:, -1, :]


LOG_STD_MIN, LOG_STD_MAX = -5.0, 2.0

class SACActor(nn.Module):
    def __init__(self, feature_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU()
        )
        self.mu_head = nn.Linear(256, action_dim)
        self.log_std_head = nn.Linear(256, action_dim)

    def forward(self, features):
        h = self.net(features)
        mu = self.mu_head(h)
        log_std = torch.clamp(self.log_std_head(h), LOG_STD_MIN, LOG_STD_MAX)
        return mu, log_std

    def sample(self, features):
        mu, log_std = self(features)
        std = log_std.exp()
        dist = torch.distributions.Normal(mu, std)
        z = dist.rsample()
        action = torch.tanh(z)
        log_prob = dist.log_prob(z) - torch.log(1 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=-1, keepdim=True)
        return action, log_prob


class SACCritic(nn.Module):
    def __init__(self, feature_dim, action_dim):
        super().__init__()
        self.q1 = nn.Sequential(
            nn.Linear(feature_dim + action_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )
        self.q2 = nn.Sequential(
            nn.Linear(feature_dim + action_dim, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, features, actions):
        x = torch.cat([features, actions], dim=-1)
        return self.q1(x), self.q2(x)


class SACModel(nn.Module):
    def __init__(self, state_dim=6, action_dim=2):
        super().__init__()
        self.extractor = TransformerFeatureExtractor(input_dim=state_dim)
        self.actor = SACActor(128, action_dim)
        self.critic = SACCritic(128, action_dim)

    def encode(self, seq):
        return self.extractor(seq)


def soft_update(target, source, tau=0.005):
    for tp, sp in zip(target.parameters(), source.parameters()):
        tp.data.copy_(tau * sp.data + (1.0 - tau) * tp.data)


In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)

    def add(self, state_seq, action, reward, next_state_seq, done):
        self.buffer.append((state_seq, action, reward, next_state_seq, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (
            torch.tensor(np.array(s), dtype=torch.float32, device=device),
            torch.tensor(np.array(a), dtype=torch.float32, device=device),
            torch.tensor(np.array(r), dtype=torch.float32, device=device).unsqueeze(-1),
            torch.tensor(np.array(ns), dtype=torch.float32, device=device),
            torch.tensor(np.array(d), dtype=torch.float32, device=device).unsqueeze(-1),
        )

    def __len__(self):
        return len(self.buffer)


class TransformerSACAgent:
    def __init__(self, state_dim=6, action_dim=2, lr=3e-4, gamma=0.99, tau=0.005):
        self.gamma = gamma
        self.tau = tau

        self.model = SACModel(state_dim=state_dim, action_dim=action_dim).to(device)
        self.target_model = SACModel(state_dim=state_dim, action_dim=action_dim).to(device)
        self.target_model.load_state_dict(self.model.state_dict())

        self.actor_opt = torch.optim.Adam(self.model.actor.parameters(), lr=lr)
        self.critic_opt = torch.optim.Adam(
            list(self.model.critic.parameters()) + list(self.model.extractor.parameters()),
            lr=lr,
        )

        self.log_alpha = torch.tensor(0.0, requires_grad=True, device=device)
        self.alpha_opt = torch.optim.Adam([self.log_alpha], lr=lr)
        self.target_entropy = -float(action_dim)

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def select_action(self, state_seq, eval_mode=False):
        s = torch.tensor(state_seq, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            feat = self.model.encode(s)
            if eval_mode:
                mu, _ = self.model.actor(feat)
                a = torch.tanh(mu)
            else:
                a, _ = self.model.actor.sample(feat)
        return a.squeeze(0).cpu().numpy()

    def update(self, replay, batch_size=64):
        s, a, r, ns, d = replay.sample(batch_size)

        with torch.no_grad():
            nfeat = self.target_model.encode(ns)
            na, nlogp = self.model.actor.sample(nfeat)
            nq1, nq2 = self.target_model.critic(nfeat, na)
            nq = torch.min(nq1, nq2) - self.alpha.detach() * nlogp
            target_q = r + (1.0 - d) * self.gamma * nq

        feat = self.model.encode(s)
        q1, q2 = self.model.critic(feat, a)
        critic_loss = F.mse_loss(q1, target_q) + F.mse_loss(q2, target_q)

        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        feat_actor = self.model.encode(s)
        ca, logp = self.model.actor.sample(feat_actor)
        cq1, cq2 = self.model.critic(feat_actor, ca)
        actor_loss = (self.alpha.detach() * logp - torch.min(cq1, cq2)).mean()

        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()

        alpha_loss = -(self.log_alpha * (logp + self.target_entropy).detach()).mean()
        self.alpha_opt.zero_grad()
        alpha_loss.backward()
        self.alpha_opt.step()

        soft_update(self.target_model, self.model, tau=self.tau)

        return {
            'critic_loss': float(critic_loss.item()),
            'actor_loss': float(actor_loss.item()),
            'alpha': float(self.alpha.item()),
        }


In [ ]:
def make_profiles(total_hours=48):
    pv = []
    prices = []
    for t in range(total_hours):
        h = t % 24
        daylight = max(0.0, math.sin((h - 6) / 12.0 * math.pi))
        pv.append(60.0 * daylight)

        if 0 <= h <= 5:
            prices.append(0.10)
        elif 6 <= h <= 16:
            prices.append(0.12)
        elif 17 <= h <= 21:
            prices.append(0.20)
        else:
            prices.append(0.14)
    return pv, prices


def build_demo_scenario():
    return [
        {'arrival_time': 7, 'initial_soc': 0.20, 'target_soc': 0.85, 'capacity': 60.0, 'duration': 8, 'departure_time': 15},
        {'arrival_time': 9, 'initial_soc': 0.30, 'target_soc': 0.80, 'capacity': 75.0, 'duration': 10, 'departure_time': 19},
        {'arrival_time': 18, 'initial_soc': 0.25, 'target_soc': 0.90, 'capacity': 70.0, 'duration': 8, 'departure_time': 26},
    ]


def encode_obs(obs):
    pv, batt_soc, price, hour, ev_need = obs
    hour_angle = 2 * math.pi * (hour / 24.0)
    return np.array([
        pv / 60.0,
        batt_soc / 150.0,
        (price - 0.10) / 0.10,
        math.sin(hour_angle),
        math.cos(hour_angle),
        min(ev_need / 300.0, 1.0),
    ], dtype=np.float32)


def train_sac(num_episodes=10, seq_len=24, batch_size=64, warmup=200):
    pv, prices = make_profiles(total_hours=48)
    env = ChargingStationEnv(pv, prices, scenario=build_demo_scenario())

    agent = TransformerSACAgent(state_dim=6, action_dim=2)
    replay = ReplayBuffer(capacity=50000)

    rewards = []
    costs = []

    for ep in range(1, num_episodes + 1):
        obs, _ = env.reset(seed=SEED + ep)
        enc = encode_obs(obs)
        state_seq = np.repeat(enc[None, :], seq_len, axis=0)

        done = False
        ep_reward = 0.0
        last_info = {}
        updates = []

        while not done:
            action = agent.select_action(state_seq, eval_mode=False)
            next_obs, reward, done, _, info = env.step(action)

            next_enc = encode_obs(next_obs)
            next_state_seq = np.concatenate([state_seq[1:], next_enc[None, :]], axis=0)

            replay.add(state_seq, action, reward, next_state_seq, float(done))

            if len(replay) >= max(batch_size, warmup):
                updates.append(agent.update(replay, batch_size=batch_size))

            state_seq = next_state_seq
            ep_reward += reward
            last_info = info

        rewards.append(ep_reward)
        costs.append(last_info.get('cumulative_cost', 0.0))

        alpha = updates[-1]['alpha'] if updates else float(agent.alpha.item())
        print(
            f"Episode {ep:02d} | Reward={ep_reward:8.2f} | Cost={costs[-1]:7.2f} | "
            f"Satisfaction={last_info.get('satisfaction', 0.0):6.2f}% | alpha={alpha:.4f}"
        )

    return rewards, costs


In [ ]:
rewards, costs = train_sac(num_episodes=10, seq_len=24)


In [ ]:
episodes = np.arange(1, len(rewards) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(episodes, rewards, marker='o')
plt.title('Reward par épisode')
plt.xlabel('Épisode')
plt.ylabel('Reward total')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(episodes, costs, marker='o', color='tab:red')
plt.title('Cost cumulé par épisode')
plt.xlabel('Épisode')
plt.ylabel('Cost')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
